# EDA 004.07 — Geospatial Analysis

**Exploring data with a geographic dimension**

## Learning Objectives
By the end of this notebook you will be able to:
- Understand coordinate systems (latitude/longitude) and common map projections
- Create point maps (scatter geo) using Plotly Express
- Build choropleth maps to colour regions by a metric
- Calculate great-circle distances using the Haversine formula
- Perform spatial aggregation (grid-based binning)

---

## Map Types at a Glance

| Map type | Shows | Best for |
|---|---|---|
| **Point map (scatter geo)** | Individual location events | Crime incidents, store locations, tweets |
| **Choropleth** | Aggregated metric per region | GDP by country, population by state |
| **Heatmap (density)** | Density of events | Bike trip origins, accident hotspots |
| **Flow / origin-destination** | Movement between places | Migration, flight routes |
| **Bubble map** | Point with size = quantity | City populations, sales by city |

---

## Popular Datasets to Practise Geospatial Analysis

| Dataset | Link | What to explore |
|---|---|---|
| NYC Taxi Trips | [Kaggle](https://www.kaggle.com/c/nyc-taxi-trip-duration) | Pickup/dropoff density maps |
| Airbnb Listings | [Inside Airbnb](http://insideairbnb.com/get-the-data/) | Price choropleth by neighbourhood |
| US Accidents | [Kaggle](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) | Accident hotspot maps |
| World Bank Development | [data.worldbank.org](https://data.worldbank.org/) | Country-level choropleths |
| OpenStreetMap | [OSM](https://www.openstreetmap.org/) | Points of interest, roads |
| Gapminder | [gapminder.org](https://www.gapminder.org/data/) | Country socioeconomic metrics |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')
%matplotlib inline

# Gapminder — built into plotly, no download required
gapminder = px.data.gapminder()
print(f'Shape: {gapminder.shape}')
print(f'Years: {sorted(gapminder.year.unique())}')
gapminder.head()

---
## 1 — Point Map (Scatter Geo)

**Reference:** [plotly.express.scatter_geo](https://plotly.com/python/scatter-plots-on-maps/)

A **scatter geo** (point map) places a marker at each (lat, lon) location.
Use `size` to encode quantity and `color` to encode a category or metric.

Plotly Express's built-in `gapminder` dataset has ISO country codes so we can use `locations='iso_alpha'` to position markers by country.

In [ ]:
# Bubble map — 2007 life expectancy, bubble size = population
df_2007 = gapminder[gapminder['year'] == 2007]

fig = px.scatter_geo(
    df_2007,
    locations='iso_alpha',
    color='continent',
    size='pop',
    hover_name='country',
    hover_data={'gdpPercap': ':.0f', 'lifeExp': ':.1f'},
    size_max=50,
    projection='natural earth',
    title='2007 — Life Expectancy Bubble Map (size = population)',
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig.update_layout(height=450)
fig.show()

---
## 2 — Choropleth Map

**Reference:** [plotly.express.choropleth](https://plotly.com/python/choropleth-maps/)

A **choropleth** fills each geographic region with a colour proportional to a metric.
It is the most common way to visualise regional statistics (population, GDP, rates, scores).

Key parameters:
- `locations` — column with ISO-3 codes, FIPS codes, or country names
- `locationmode` — `'ISO-3'`, `'USA-states'`, `'country names'`
- `color` — the numeric column to visualise
- `color_continuous_scale` — colormap (e.g. `'Viridis'`, `'RdYlGn'`)

In [ ]:
fig = px.choropleth(
    df_2007,
    locations='iso_alpha',
    color='lifeExp',
    hover_name='country',
    hover_data={'gdpPercap': ':,.0f'},
    color_continuous_scale='RdYlGn',
    range_color=(40, 85),
    projection='natural earth',
    title='2007 — Life Expectancy by Country'
)
fig.update_layout(height=450, coloraxis_colorbar_title='Life Exp.')
fig.show()

In [ ]:
# Animated choropleth — show change over time
fig = px.choropleth(
    gapminder,
    locations='iso_alpha',
    color='gdpPercap',
    hover_name='country',
    animation_frame='year',
    color_continuous_scale='Blues',
    range_color=(0, 50000),
    projection='natural earth',
    title='GDP per Capita 1952-2007 (animated)'
)
fig.update_layout(height=450)
fig.show()

---
## 3 — Haversine Distance

**Reference:** [Wikipedia — Haversine formula](https://en.wikipedia.org/wiki/Haversine_formula)

The **Haversine formula** calculates the great-circle distance between two points on a sphere (Earth) given their latitude and longitude:

$$d = 2r \arcsin\!\left(\sqrt{\sin^2\!\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\!\left(\frac{\Delta\lambda}{2}\right)}\right)$$

where $r \approx 6371$ km, $\phi$ = latitude, $\lambda$ = longitude (in radians).

This is the **as-the-crow-flies** distance, not road distance.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance in km between two (lat, lon) points."""
    R = 6371  # Earth radius in km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi   = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Sample city coordinates
cities = {
    'New York':  (40.7128, -74.0060),
    'London':    (51.5074,  -0.1278),
    'Tokyo':     (35.6762, 139.6503),
    'Sydney':   (-33.8688, 151.2093),
    'Mumbai':    (19.0760,  72.8777),
    'São Paulo': (-23.5505, -46.6333),
}

# Build a distance matrix
city_names = list(cities.keys())
n = len(city_names)
dist_matrix = pd.DataFrame(np.zeros((n, n)), index=city_names, columns=city_names)

for c1 in city_names:
    for c2 in city_names:
        lat1, lon1 = cities[c1]
        lat2, lon2 = cities[c2]
        dist_matrix.loc[c1, c2] = haversine_km(lat1, lon1, lat2, lon2)

print('Great-circle distances (km):')
print(dist_matrix.round(0).astype(int))

fig, ax = plt.subplots(figsize=(8, 5))
mask = np.triu(np.ones(dist_matrix.shape, dtype=bool))
sns.heatmap(dist_matrix, mask=mask, annot=True, fmt='.0f',
            cmap='YlOrRd', ax=ax, linewidths=0.5,
            annot_kws={'size': 9})
ax.set_title('City Distance Matrix (km) — Haversine')
plt.tight_layout()
plt.show()

---
## 4 — Spatial Aggregation (Grid Binning)

**Reference:** [numpy.histogram2d](https://numpy.org/doc/stable/reference/generated/numpy.histogram2d.html)

When you have many location points (e.g. thousands of GPS records), individual points overlap.
**Grid binning** divides the map into a grid and counts observations per cell — creating a spatial density heatmap.

This is the 2D equivalent of a histogram, applied to geographic coordinates.

In [ ]:
np.random.seed(42)

# Simulate 2000 GPS points — clustered around 3 hotspots
hotspots = [(-74.0, 40.73), (-73.97, 40.76), (-73.99, 40.70)]  # NYC-ish
lons, lats = [], []
for lon, lat in hotspots:
    n = 600
    lons.extend(np.random.normal(lon, 0.02, n))
    lats.extend(np.random.normal(lat, 0.015, n))
# Add some random scattered points
lons.extend(np.random.uniform(-74.05, -73.90, 200))
lats.extend(np.random.uniform(40.68, 40.78, 200))

lons = np.array(lons)
lats = np.array(lats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw scatter — overplotted
axes[0].scatter(lons, lats, alpha=0.05, s=5, color='steelblue')
axes[0].set_title(f'Raw Points ({len(lons):,}) — overplotted')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

# 2D histogram (grid binning)
h, xedges, yedges = np.histogram2d(lons, lats, bins=30)
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
im = axes[1].imshow(h.T, origin='lower', extent=extent,
                     aspect='auto', cmap='hot_r', interpolation='bilinear')
plt.colorbar(im, ax=axes[1], label='Count per cell')
axes[1].set_title('Grid Density Map (30×30 bins)')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.show()

---
## 5 — Interactive Map with Plotly (Mapbox / OpenStreetMap)

**Reference:** [plotly.express.scatter_mapbox](https://plotly.com/python/scattermapbox/)

For real-world data with actual latitude/longitude columns, `scatter_mapbox` overlays points on an interactive OpenStreetMap tile layer.

In [ ]:
# Create a DataFrame from the simulated GPS points
df_points = pd.DataFrame({
    'lat': lats,
    'lon': lons,
    'category': np.random.choice(['Type A', 'Type B', 'Type C'], size=len(lons))
})

fig = px.scatter_mapbox(
    df_points,
    lat='lat',
    lon='lon',
    color='category',
    zoom=11,
    height=450,
    title='Simulated GPS Points on OpenStreetMap',
    mapbox_style='open-street-map',
    opacity=0.4,
    size_max=6
)
fig.show()

---
## Key Takeaways

- Use **scatter geo / bubble maps** for individual location events — encode extra dimensions with size and colour
- Use **choropleth maps** for regional aggregates (country, state, postcode level)
- Use **Plotly Express** for interactive maps that need no backend server
- The **Haversine formula** gives great-circle (as-the-crow-flies) distances from lat/lon
- **Grid binning (2D histogram)** resolves overplotting for dense point clouds
- Always validate lat/lon ranges: latitude ∈ [−90, 90], longitude ∈ [−180, 180]

---
## Further Reading

- [Kaggle — Geospatial Analysis Course](https://www.kaggle.com/learn/geospatial-analysis) (uses geopandas + folium)
- [Plotly Python Maps](https://plotly.com/python/maps/)
- [GeoPandas documentation](https://geopandas.org/en/stable/)
- [Folium — Interactive Leaflet Maps in Python](https://python-visualization.github.io/folium/)
- [Geographic Data Science with Python (free online)](https://geographicdata.science/book/) — Rey et al.